# EDA Summary and Modelling Implications

This notebook synthesises the key findings from all EDA notebooks in this project and translates them into concrete requirements for the modelling stage.

**EDA notebooks covered:**
- `AUS Enrollments EDA.ipynb`
- `UK Enrollments EDA.ipynb`
- `AUS Funding EDA.ipynb` / `AnnualFundingAUS2019-2026_EDA.ipynb`
- `Employment by Industry EDA.ipynb`
- `AUS_Enrollments_Funding_Multivariate_EDA.ipynb`
- `Multivariate EDA UK AUS Enrollments.ipynb`

## 1. Distribution and Transformation

| Series | Raw skewness | Log(1+x) skewness | Decision |
|--------|-------------|-------------------|----------|
| AUS enrollments (category-year) | ≈ 1.1 | ≈ 0.3 | Log-transform for regression |
| UK enrollments (category-year) | ≈ 2.0 | ≈ 0.5 | Log-transform for regression |
| AUS funding (per-unit rates) | ≈ 0.5 | ≈ 0.2 | Mild; log acceptable but not required |
| Employment (category quarterly) | ≈ 1.1 | ≈ 0.4 | Log-transform if pooled across categories |

**Implication:** Use `log(1 + Enrollments)` as the outcome variable in any regression. This ensures that residuals are approximately normally distributed and that estimated coefficients have a percentage-change interpretation.

## 2. Simpson's Paradox — Fixed Effects Are Required

Simpson's Paradox was detected in **both countries** and in the **employment** series:

- **AUS enrollments:** Aggregate trend positive; Creative Arts and Management & Commerce negative within category.
- **UK enrollments:** Aggregate trend positive; Creative Arts, Natural & Physical Sciences, and Environment negative within category.
- **Employment:** Aggregate trend strongly positive (r = 0.98); Information Technology and Environment & Related negative within category.
- **AUS funding × enrollments:** Pooled r(TotalFunding, Enrollments) positive, driven by category size. Several categories show a *negative* within-category relationship.

**Implication:** All regression models **must** include **category fixed effects**. Pooled OLS without fixed effects will produce biased, misleading slope coefficients. Within-estimator (demeaned) or first-differences specifications are appropriate.

## 3. Structural Break — JRG 2021 Indicator

The Job-Ready Graduates (JRG) reform commenced in **2021**, creating a visible step-change in:
- Student contribution rates (focal-value bunching analysis confirms discontinuity)
- Commonwealth contribution levels (cluster-year line plots show the break)
- Enrollment growth rates post-2021 (category-level shifts visible in the contingency analysis)

The post-JRG distribution of student contribution rates is **wider and higher** than the pre-JRG distribution, confirming that the reform repriced courses across clusters, not just at the extremes.

**Implication:** Any model pooling 2019–2024 data must include a `PostJRG` binary indicator (= 1 for 2021 onwards) or a piecewise linear trend with a knot at 2021. A regression without this term will confound the pre- and post-reform periods and produce a biased time coefficient. Consider also a `PostJRG × Category` interaction to test whether the enrollment response to the reform differed by field.

## 4. COVID-Period Disruption — Dummy or Exclusion

The employment dataset has **7 missing quarterly values** concentrated in **2020 Q2 – 2021 Q3**. The missingness pattern is systematic (ABS collection disruption), not random. The total employment series also shows a visible dip and recovery across this window.

The rolling-average analysis confirms that the COVID dip is a transient shock, not a structural change — the 4-quarter rolling mean returns to the pre-2020 trend line for most categories by 2022.

**Implication:** In any time-series or panel model that includes employment data:
- Include a `COVID` dummy (= 1 for 2020–2021 quarters) to absorb the shock.
- Do **not** impute missing values as zero or via simple interpolation — the shock was real and the recovered level is what matters.
- Alternatively, restrict the employment analysis to pre-2020 and post-2022 if a clean trend estimate is required.

## 5. Spurious Size Correlation — Within-Category Analysis Only

The positive aggregate correlation between funding and enrollments is driven by **category size**: large fields (Health, Engineering, Management & Commerce) receive more total funding *and* enroll more students simply because they are larger. This is not evidence of a causal funding → enrollment effect.

The residual-thinking analysis confirms that Year alone explains only ~10–15% of enrollment variance. After removing the time trend, persistent category-level offsets dominate the residuals — category fixed effects are not optional.

**Implication:** Use **within-category** variation only. This means:
- Demean by category before running any regression (or use the `C(Category)` fixed-effects specification in statsmodels).
- Use **enrollment growth rates** or year-on-year changes rather than levels.
- The contingency table (Category × Year) shows a balanced 11 × 6 panel — no sparse cells in the merged dataset, so all categories contribute to the within estimator.

## 6. UK as a Difference-in-Differences Control

The UK data (HESA enrollments 2014/15 – 2022/23) provides a potential control group for a difference-in-differences (DiD) design:
- UK did **not** implement JRG or an equivalent reform.
- UK enrollment trends by field-of-education category are observable over the same period.
- Both countries show Simpson's Paradox, suggesting structural similarities.

**Known threats to parallel trends:**
- UK academic years (2020/21) introduce a ~6-month calendar misalignment relative to AUS calendar years.
- UK data uses HESA subject categories mapped to 10 FOE groups — some mapping approximation is unavoidable.
- Both countries were hit by COVID, but the timing and severity of enrollment disruptions may differ.

**Implication:** A DiD specification of the form:

> `log(Enrollments) ~ PostJRG × AUS + Category FE + Year FE + COVID dummy`

is the appropriate model. The parallel-trends assumption should be tested visually (pre-2021 trend lines) before interpreting the DiD coefficient as causal.

## 7. Small N Warning

The merged AUS panel contains **~66 observations** (11 categories × 6 years). After absorbing category fixed effects, the within-category identification rests on **6 time points per category**. This severely limits statistical power:

- Standard errors on the `PostJRG` coefficient will be wide.
- Interaction terms (`PostJRG × Category`) will be even less precisely estimated.
- Non-parametric tests (Wilcoxon, permutation) may be more appropriate than t-tests for within-category comparisons.

**Implication:** Report confidence intervals, not just point estimates. Do not over-interpret marginal significance. Consider augmenting with the UK comparison to increase effective sample size via the DiD design.

## 8. Recommended Model Specification

Based on all EDA findings, the recommended baseline model is:

```python
import statsmodels.formula.api as smf
import numpy as np

# Merged panel: AUS + UK (long format)
# Columns: log_Enrollments, PostJRG, AUS_dummy, Year, Category, COVID

model = smf.ols(
    'log_Enrollments ~ PostJRG * AUS_dummy + C(Category) + C(Year) + COVID',
    data=panel
).fit(cov_type='HC3')  # Heteroskedasticity-robust SEs
```

**Key choices and their EDA justifications:**

| Choice | EDA finding that justifies it |
|--------|------------------------------|
| `log_Enrollments` outcome | Right-skewed distribution; log reduces skewness from ~1.1 to ~0.3 |
| `C(Category)` fixed effects | Simpson's Paradox and residual thinking both show large persistent offsets |
| `C(Year)` fixed effects | Removes common time shocks (enrollment growth trend, COVID) |
| `PostJRG` indicator | Focal-value analysis confirms structural break at 2021 |
| `PostJRG × AUS_dummy` | DiD coefficient — isolates JRG effect from common UK/AUS trends |
| `COVID` dummy | 7 systematic missing employment values; visible dip in enrollment series |
| `HC3` robust SEs | Levene's test confirms heteroskedasticity across categories |